# Data assimilation in the latent space

## Experiment output folder

In [ ]:
from pathlib import Path

# Define the subfolder path for outputs
output_dir = Path("output/ETNA2018")

## Open prior and observation datasets

In [ ]:
import xarray as xr
import numpy as np

fname_obs = "data/20181224_1800_Meteosat-11_Etna_VPRoutput.nc"

prior = xr.open_dataset("output/ETNA2018/prior.nc")
obs = xr.open_dataset(fname_obs)

In [ ]:
N_ens = prior.sizes['ens']
latent_dim = prior.sizes['latent_dim']
print(f"Ensemble size: {N_ens} --- Latent space size: {latent_dim}")

## Get observations

In [ ]:
valid = (obs["plume_mask"].values > 0) & (obs["mass"].values > 0)
y_obs = obs["mass"].values[valid].astype(np.float64)
lat_obs = obs["latitude"].values[valid].astype(np.float64)
lon_obs = obs["longitude"].values[valid].astype(np.float64)

In [ ]:
N_obs = len(y_obs)
print(f"Number of observations: {N_obs}")

In [ ]:
# Required for interpolations
lat_points = xr.DataArray(lat_obs, dims="obs")
lon_points = xr.DataArray(lon_obs, dims="obs")

## Get the prior ensemble

In [ ]:
Z = prior["z"].values.astype(np.float64)
X = prior["samples"]

Y_xr = X.interp(
    lat=lat_points,
    lon=lon_points,
    method="linear"
)
Y = Y_xr.values.astype(np.float64)

print("Y shape:", Y.shape)

In [ ]:
print("NaNs in Y:", np.isnan(Y).sum())

## Compute anomalies

In [ ]:
Z_mean = Z.mean(axis=0)
Y_mean = Y.mean(axis=0)

Z_anom = Z - Z_mean
Y_anom = Y - Y_mean

In [ ]:
assert Z_anom.shape == (N_ens, latent_dim), "Wrong dimensions for Z_anom"
assert Y_anom.shape == (N_ens, N_obs), "Wrong dimensions for Y_anom"

## Observation-error covariance

In [ ]:
# 50% error observation is assumed with a minimim of
# 0.1 g/m2 (typical satellite detection limit)
obs_error_fraction = 0.50
obs_error_std = obs_error_fraction * y_obs
obs_error_std[obs_error_std<0.1] = 0.1 # Apply a minimum error (satellite detection limit)
R_diag = obs_error_std**2

print("R_diag shape:", R_diag.shape)
print("error std range:", obs_error_std.min(), obs_error_std.max())
print("R diagonal range:", R_diag.min(), R_diag.max())

## ETKF

#### Normalized, error-weighted observation anomalies

In [ ]:
# Ensemble anomaly of an observation, normalized by its observation uncertainty:
sigma_R = np.sqrt(R_diag)
A = Y_anom / np.sqrt(N_ens - 1)
A_tilde = A / sigma_R[None, :]

#### Ensemble-space matrix

In [ ]:
C = np.eye(N_ens) + A_tilde @ A_tilde.T

In [ ]:
print("condition number:", np.linalg.cond(C))

#### Innovations

In [ ]:
innovation = y_obs - Y_mean

innovation_tilde = innovation / sigma_R

In [ ]:
print("innovation shape:", innovation.shape)
print("innovation_tilde shape:", innovation_tilde.shape)

In [ ]:
print("innovation_tilde mean:", innovation_tilde.mean())
print("innovation_tilde std:", innovation_tilde.std())
print("innovation_tilde min:", innovation_tilde.min())
print("innovation_tilde max:", innovation_tilde.max())

#### Solve the ensemble-space system

In [ ]:
rhs = A_tilde @ innovation_tilde

print("rhs shape:", rhs.shape)
print("rhs mean:", rhs.mean())
print("rhs std:", rhs.std())

In [ ]:
w = np.linalg.solve(C, rhs)

print("w shape:", w.shape)
print("w mean:", w.mean())
print("w std:", w.std())
print("w min:", w.min())
print("w max:", w.max())

#### Analysis mean in latent space

In [ ]:
Z_mean_analysis = Z_mean + Z_anom.T @ w / np.sqrt(N_ens - 1)

In [ ]:
print("Z_mean prior:   ", Z_mean)
print("Z_mean analysis:", Z_mean_analysis)

#### ETKF transform matrix

In [ ]:
eigvals, eigvecs = np.linalg.eigh(C)

print("eigenvalue range:", eigvals.min(), eigvals.max())

In [ ]:
T = (
    eigvecs
    @ np.diag(1.0 / np.sqrt(eigvals))
    @ eigvecs.T
)

In [ ]:
check = T @ C @ T.T

print(
    "T C T.T error:",
    np.max(np.abs(check - np.eye(N_ens)))
)

In [ ]:
Z_anom_analysis = T @ Z_anom
Z_analysis = Z_mean_analysis + Z_anom_analysis

## Summary

In [ ]:
Pz_prior = np.cov(Z, rowvar=False)
Pz_analysis = np.cov(Z_analysis, rowvar=False)

In [ ]:
variance_prior = np.trace(Pz_prior)
variance_analysis = np.trace(Pz_analysis)

print("Total prior variance:   ", variance_prior)
print("Total analysis variance:", variance_analysis)
print("Variance ratio:",variance_analysis / variance_prior)

In [ ]:
eig_prior = np.linalg.eigvalsh(Pz_prior)
eig_analysis = np.linalg.eigvalsh(Pz_analysis)

print("Smallest prior eigenvalue:   ", eig_prior.min())
print("Smallest analysis eigenvalue:", eig_analysis.min())

## Save posterior

In [ ]:
posterior_latent = xr.Dataset(
    data_vars={
        "z": (
            ("ens", "latent_dim"),
            Z_analysis.astype(np.float32)
        ),
        "z_mean": (
            ("latent_dim",),
            Z_mean_analysis.astype(np.float32)
        ),
    },
    coords={
        "ens": np.arange(N_ens),
        "latent_dim": np.arange(latent_dim),
    },
)

posterior_latent.to_netcdf(output_dir / "posterior-latent-etkf.nc")